# 워크 워크 (Work Walk) - 직장인 산책 코스 추천 Agent
설계서(Agent_설계서_4조) 중 Tool 구현 부분만 발췌하여 기존 Basic Agent 노트북과 동일한 형식으로 구현합니다.

### API Key

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

for key in ["KAKAO_API_KEY", "T1_API_KEY", "KPA_API_KEY", "OPENROUTE_API_KEY"]:
    print(f".env 내 {key}가 환경변수에 할당됐습니다:", os.environ[key][:5]+"*****")

### Install package

In [ ]:
!pip install -q langchain langchain-openai langchain-community requests

# Tool 구현 (설계서 2.5 기능→API 매핑)

## 0. 보조 함수 (API 불필요)
실제 계산 로직은 `utils.py`에 있습니다(테스트 스크립트 `test_agent.py`와 공유). 여기서는 그 함수를 가져와 `tool()`로 감싸 Agent가 호출할 수 있는 Tool로 등록합니다.
`calc_time_budget`, `straight_distance`는 Agent가 직접 호출하는 Tool이고, `latlon_to_grid`는 `check_weather` 내부에서만 쓰는 순수 변환 함수라 Tool로 등록하지 않습니다.

In [ ]:
from langchain.tools import tool
from datetime import datetime, timedelta

# 순수 계산 로직(외부 API 불필요)은 utils.py에 모아뒀고, test_agent.py와 이 노트북이 함께 가져다 쓴다.
# KST/datetime/timedelta는 뒤에 나올 check_weather 셀에서도 그대로 재사용한다.
from utils import (
    KST,
    calc_time_budget as _calc_time_budget,
    straight_distance as _straight_distance,
    latlon_to_grid,  # check_weather 내부에서만 쓰는 헬퍼라 Tool로 감싸지 않고 함수 그대로 둔다
)

# tool(fn)은 @tool 데코레이터를 함수 형태로 적용한 것과 동일하다.
# fn의 이름·타입힌트·docstring을 그대로 읽어 Tool 스키마를 만들며, 이 docstring이 곧
# LLM에게 "이 Tool을 언제 호출해야 하는지" 알려주는 설명이 된다.
calc_time_budget = tool(_calc_time_budget)
straight_distance = tool(_straight_distance)

# 자체 점검 (외부 API 없이 검증 가능한 로직만 확인)
# - calc_time_budget: 자정에 가까운 종료시각을 줘도 음수로 튀지 않고 0 이상인지
# - straight_distance: 같은 좌표 두 개를 넣으면 거리 0이 나오는지 (하버사인 공식 기본 검증)
# - latlon_to_grid: 서울시청 좌표를 기상청 공식 예시 격자값 (60, 127)로 정확히 변환하는지
assert calc_time_budget.invoke({"end_time": "23:59", "start_time": "00:01", "coffee_min": 10}) >= 0
assert straight_distance.invoke({"lat1": 37.5665, "lon1": 126.9780, "lat2": 37.5665, "lon2": 126.9780}) == 0
assert latlon_to_grid(37.5665, 126.9780) == (60, 127)  # 서울시청 기준 공식 예시값
print("보조 함수 self-check 통과")

## 1. 카카오 로컬 - 주소→좌표 (geocode)

In [ ]:
import requests

# 카카오 로컬 API는 인증키를 HTTP 헤더 Authorization에 "KakaoAK {키}" 형식으로 담아 보낸다.
# ①②③⑥(geocode/find_cafes/find_parks/reverse_geocode) 네 개 Tool이 이 헤더 하나를 공유한다
# - 카카오는 REST API 키 한 개로 모든 로컬 API를 쓸 수 있기 때문.
KAKAO_HEADERS = {"Authorization": f"KakaoAK {os.environ['KAKAO_API_KEY']}"}

@tool
def geocode(address: str) -> dict:
    """주소를 좌표로 변환합니다. 산책 코스의 출발점(회사 주소)을 구할 때 가장 먼저 호출하세요.
    반환값의 x=경도, y=위도입니다. 이후 모든 Tool의 출발점으로 사용됩니다.

    Args:
        address: 지번 또는 도로명 주소 (예: "경기 성남시 분당구 판교역로 231")
    """
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    resp = requests.get(url, headers=KAKAO_HEADERS, params={"query": address})
    resp.raise_for_status()  # 4xx/5xx면 여기서 예외를 던져 LangChain이 Tool 실행 에러로 처리하게 한다
    doc = resp.json()["documents"][0]  # 검색 결과 중 정확도가 가장 높은 첫 번째 주소만 사용
    return {"x": float(doc["x"]), "y": float(doc["y"]), "address_name": doc["address_name"]}

## 2. 카카오 로컬 - 카페 찾기 (find_cafes)

In [ ]:
@tool
def find_cafes(x: float, y: float, radius_m: int = 300) -> list:
    """좌표 주변 카페 목록을 검색합니다. 카카오 카테고리 코드 CE7(카페)을 사용합니다.
    x=경도, y=위도입니다. 비가 오면 반경을 300m로 줄이는 판단은 Agent 몫입니다.

    Args:
        x: 중심 좌표 경도
        y: 중심 좌표 위도
        radius_m: 검색 반경(m), 최대 20000
    """
    # 카페는 카카오가 제공하는 전용 카테고리 코드(CE7)가 있어서 키워드 검색보다 정확도가 높다
    # - "카페"라는 이름이 들어간 다른 업종(예: PC방 "○○카페")이 섞이지 않는다.
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    params = {"category_group_code": "CE7", "x": x, "y": y, "radius": radius_m, "sort": "distance"}
    resp = requests.get(url, headers=KAKAO_HEADERS, params=params)
    resp.raise_for_status()
    docs = resp.json()["documents"]
    # 카카오 로컬 정책상 검색 결과(좌표·주소)를 DB에 영구 저장하면 안 되므로,
    # 장기 기억에는 place_id(장소 고유 ID)만 남기고 좌표는 필요할 때마다 다시 검색한다는 전제로
    # 여기서도 좌표를 응답에 포함시키되 "저장용"이 아니라 "이번 요청에서 즉시 쓸 값"으로만 취급한다.
    return [
        {"name": d["place_name"], "x": float(d["x"]), "y": float(d["y"]), "distance_m": int(d["distance"]), "place_id": d["id"]}
        for d in docs
    ]

## 3. 카카오 로컬 - 공원 찾기 (find_parks)

In [ ]:
@tool
def find_parks(x: float, y: float, radius_m: int = 500) -> list:
    """좌표 주변 공원을 검색합니다. 공원은 카테고리 코드가 없어 키워드("공원")로 검색합니다.
    x=경도, y=위도이며 반경·정렬(거리순) 지정이 가능합니다.

    Args:
        x: 중심 좌표 경도
        y: 중심 좌표 위도
        radius_m: 검색 반경(m)
    """
    # find_cafes와 달리 공원은 카카오가 제공하는 전용 카테고리 코드가 없다.
    # 그래서 category.json 대신 keyword.json으로, "공원"이라는 문자열 검색을 쓴다.
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    params = {"query": "공원", "x": x, "y": y, "radius": radius_m, "sort": "distance"}
    resp = requests.get(url, headers=KAKAO_HEADERS, params=params)
    resp.raise_for_status()
    docs = resp.json()["documents"]
    return [
        {"name": d["place_name"], "x": float(d["x"]), "y": float(d["y"]), "distance_m": int(d["distance"]), "place_id": d["id"]}
        for d in docs
    ]

## 4. TMAP - 도보 경로/시간 계산 (walk_route)

In [ ]:
# TMAP은 카카오와 달리 인증키를 헤더 "appKey"에 그대로 넣는다 (헤더 이름/형식이 카카오와 다름)
T1_HEADERS = {"appKey": os.environ["T1_API_KEY"]}

@tool
def walk_route(start_x: float, start_y: float, end_x: float, end_y: float) -> dict:
    """출발-도착 좌표 간 도보 경로의 총 거리(m)·소요시간(초)을 계산합니다.
    회사→카페→회사처럼 경유지가 있는 왕복 코스는 구간별로 나눠 호출한 뒤 합산하세요.
    좌표는 x=경도, y=위도입니다(startX=경도, startY=위도).

    Args:
        start_x: 출발지 경도
        start_y: 출발지 위도
        end_x: 도착지 경도
        end_y: 도착지 위도
    """
    url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"
    body = {
        "startX": start_x, "startY": start_y,
        "endX": end_x, "endY": end_y,
        "startName": "출발", "endName": "도착",  # TMAP 필수 파라미터(안내 문구용, 값 자체는 임의로 둬도 무방)
        "reqCoordType": "WGS84GEO", "resCoordType": "WGS84GEO",  # 입출력 모두 위경도 좌표계 사용을 명시
    }
    # 다른 Tool과 달리 GET이 아니라 POST로 JSON 바디를 보낸다 (TMAP 보행자 경로 API의 요구사항)
    resp = requests.post(url, headers=T1_HEADERS, json=body)
    resp.raise_for_status()
    # 응답은 GeoJSON FeatureCollection이다. features[0]은 출발점을 나타내는 Point 타입 feature인데
    # 그 properties 안에 경로 "전체"의 총 거리/시간이 들어있다.
    # (뒤에 이어지는 feature들은 구간별 세부 좌표(LineString)라 여기서는 쓰지 않는다.)
    props = resp.json()["features"][0]["properties"]
    return {"total_distance_m": int(props["totalDistance"]), "total_time_sec": int(props["totalTime"])}

## 4-1. OpenRouteService - 목적지 없는 순환 도보 코스 (walk_route_roundtrip)
walk_route(TMAP)는 출발-도착 두 좌표가 필요하지만, "그냥 20분만 걷고 싶다"처럼 목적지가 없는 요청(설계서 S4류 시나리오)에는 출발점 하나와 원하는 거리만으로 순환 코스를 만들어주는 OpenRouteService의 round_trip 옵션을 씁니다.

In [ ]:
# OpenRouteService는 헤더 Authorization에 키를 그대로 넣는다 (카카오의 "KakaoAK" 같은 접두사 없음)
ORS_HEADERS = {"Authorization": os.environ["OPENROUTE_API_KEY"], "Content-Type": "application/json"}

@tool
def walk_route_roundtrip(start_x: float, start_y: float, length_m: int = 1000) -> dict:
    """출발지에서 원하는 거리만큼 걷고 다시 출발지로 돌아오는 순환(왕복) 도보 코스를 계산합니다.
    목적지를 정하지 않고 "그냥 20분만 걷고 싶다"처럼 목적지가 없는 요청에 사용하세요.
    walk_route(TMAP)는 출발-도착 두 점이 필요하지만, 이 Tool은 출발점 하나와 원하는 거리만 있으면 됩니다.

    Args:
        start_x: 출발지 경도
        start_y: 출발지 위도
        length_m: 원하는 총 도보 거리(m)
    """
    url = "https://api.openrouteservice.org/v2/directions/foot-walking/geojson"
    body = {
        "coordinates": [[start_x, start_y]],  # round_trip은 출발점 좌표 하나만 필요 (도착점 없음)
        "options": {"round_trip": {"length": length_m, "points": 3}},  # points: 순환 경로를 만드는 경유점 개수
    }
    resp = requests.post(url, headers=ORS_HEADERS, json=body)
    resp.raise_for_status()
    # TMAP과 달리 실제 이동거리/시간은 properties.summary에 바로 들어있다
    summary = resp.json()["features"][0]["properties"]["summary"]
    return {"total_distance_m": int(summary["distance"]), "total_time_sec": int(summary["duration"])}

## 5. 기상청 API허브 - 날씨 확인 (check_weather)

In [ ]:
# 기상청 API허브는 헤더가 아니라 URL 쿼리 파라미터 "authKey"로 인증한다 (요청마다 params에 직접 포함)
KPA_AUTH_KEY = os.environ["KPA_API_KEY"]

@tool
def check_weather(x: float, y: float) -> dict:
    """좌표의 초단기예보(강수형태, 기온)를 조회합니다. 비/폭염 여부 분기의 근거로 사용하세요.
    x=경도, y=위도입니다. 내부적으로 위경도를 기상청 격자 좌표로 변환해 조회합니다.

    Args:
        x: 조회 지점 경도
        y: 조회 지점 위도
    """
    # 기상청은 위경도가 아니라 격자(nx, ny)로 조회한다. latlon_to_grid는 (위도, 경도) 순서를 받으므로
    # x=경도, y=위도인 이 함수의 인자 순서를 뒤집어서(y, x) 넘긴다.
    nx, ny = latlon_to_grid(y, x)
    # 초단기예보는 매시 30분에 발표되고, 발표 후 10분 뒤부터 조회 가능하다.
    # "지금 - 45분"을 기준시로 잡으면 항상 이미 발표가 끝난 가장 최근 회차를 안전하게 가리킨다.
    base = datetime.now(KST) - timedelta(minutes=45)
    url = "https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getUltraSrtFcst"
    params = {
        "authKey": KPA_AUTH_KEY, "pageNo": 1, "numOfRows": 60, "dataType": "JSON",
        "base_date": base.strftime("%Y%m%d"), "base_time": base.strftime("%H30"),
        "nx": nx, "ny": ny,
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    items = resp.json()["response"]["body"]["items"]["item"]
    # 응답은 (여러 예보 시각) x (여러 카테고리: PTY/T1H/POP/...) 조합이 한 리스트에 다 섞여 온다.
    # 가장 이른 예보 시각(첫 item의 fcstTime) 한 시점만 골라 카테고리별 값을 딕셔너리로 재구성한다.
    first_time = items[0]["fcstTime"]
    values = {item["category"]: item["fcstValue"] for item in items if item["fcstTime"] == first_time}
    return {
        "precipitation_type": values.get("PTY"),  # 강수형태 코드: 0=없음, 1=비, 2=비/눈, 3=눈, 4=소나기
        "temperature_c": float(values.get("T1H", 0)),  # 기온(섭씨)
    }

## 6. 카카오 로컬 - 좌표→주소 (reverse_geocode)

In [ ]:
@tool
def reverse_geocode(x: float, y: float) -> str:
    """좌표를 사람이 읽는 주소로 변환합니다. 최종 답변에서 "○○로 12"처럼 장소를 안내할 때 사용하세요.
    x=경도, y=위도입니다. (①과 같은 키를 사용)

    Args:
        x: 경도
        y: 위도
    """
    url = "https://dapi.kakao.com/v2/local/geo/coord2address.json"
    resp = requests.get(url, headers=KAKAO_HEADERS, params={"x": x, "y": y})
    resp.raise_for_status()
    doc = resp.json()["documents"][0]
    road = doc.get("road_address")  # 도로명주소가 있으면 우선 사용 (사람이 더 읽기 편함)
    return road["address_name"] if road else doc["address"]["address_name"]  # 없으면 지번주소로 대체

## Agent 생성

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# 설계서 2.3 메인 모델 항목: gpt-5-mini (Tool calling·Structured Output 지원, 비용 낮음)
model = init_chat_model("gpt-5-mini")

# 7개 API Tool + 2개 순수 함수 Tool을 모두 등록한다.
# 어떤 상황에 어떤 Tool을 어떤 순서로 부를지는 개발자가 코드로 고정하지 않고,
# 모델이 사용자 요청·날씨·검색 결과를 보고 스스로 판단하게 한다 (설계서 "챗봇과의 차이").
tools = [
    geocode, find_cafes, find_parks, walk_route, walk_route_roundtrip,
    check_weather, reverse_geocode, calc_time_budget, straight_distance,
]

agent = create_agent(
    model=model,
    tools=tools,
    # 설계서 2.3의 System Prompt 원문 그대로 사용:
    # 거리·시간을 모델이 추측하지 않고 반드시 Tool 결과에만 근거하도록 못박아 환각을 억제한다.
    system_prompt=(
        "당신은 직장인 점심 산책 코스를 추천하는 Agent입니다. "
        "사용자의 시간 제약과 실시간 날씨를 반드시 Tool로 확인한 뒤 Tool 결과에 근거해서만 코스를 제안하세요. "
        "정보가 부족하면 적절한 Tool을 추가로 호출하세요."
    ),
)

In [ ]:
agent

## 실행 테스트 (시나리오 1: 맑은 날씨, 커피 한 잔, 남은 점심시간 30분)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "회사 주소는 경기 성남시 분당구 판교역로 231이고, 남은 점심시간은 30분이야. 소화도 시킬 겸 커피 한 잔 사러 걸을까?"}]},
)

In [ ]:
response["messages"][-1].pretty_print()